# Part A — Subword tokenization
## Exercise 1 — Apply BPE merges


In [ ]:
def bpe_tokenize_word(word, merges, base_vocab, unk_token="[UNK]"):
    """Split one word by applying BPE merges in their learned order. Returns a list of tokens."""
    tokens = [c if c in base_vocab else unk_token for c in word]
    for (a, b), merged in merges:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == a and tokens[i+1] == b:
                tokens = tokens[:i] + [merged] + tokens[i+2:]
            else:
                i += 1
    return tokens


## Exercise 2 — WordPiece longest match


In [ ]:
def wordpiece_tokenize_word(word, vocab, unk_token="[UNK]", prefix="##"):
    """Split one word by greedy longest match. Returns a list of tokens, or [unk_token] for the whole word."""
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        match = None
        while end > start:
            sub = word[start:end]
            if start > 0:
                sub = prefix + sub
            if sub in vocab:
                match = sub
                break
            end -= 1
        
        if match is None:
            return [unk_token]
            
        tokens.append(match)
        start = end
    return tokens


## Exercise 3 — How vocabulary size changes sequence length


In [ ]:
def subword_stats(tokenizer, texts):
    """Return {"fertility": tokens per word, "split_rate": fraction of words with >1 token, "unk_rate": [UNK] share}."""
    n_words = n_tokens = n_split = n_unk = 0
    for enc in tokenizer.encode_batch(texts):
        word_ids = [wid for wid in enc.word_ids if wid is not None]
        if not word_ids: 
            continue
            
        n_tokens += len(enc.tokens)
        n_words += max(word_ids) + 1
        
        from collections import Counter
        counts = Counter(word_ids)
        n_split += sum(1 for v in counts.values() if v > 1)
        n_unk += sum(1 for t in enc.tokens if t == "[UNK]")
        
    if n_words == 0: 
        return {"fertility": 0, "split_rate": 0, "unk_rate": 0}
        
    return {
        "fertility": n_tokens / n_words, 
        "split_rate": n_split / n_words, 
        "unk_rate": n_unk / n_tokens
    }


## Exercise 4 — One sentence, three pretrained tokenizers, three languages


In [ ]:
def tokens_per_word(tokenizer, text):
    """Tokens produced by tokenizer.tokenize(text), divided by the number of whitespace-separated words."""
    words = text.split()
    if not words:
        return 0.0
    tokens = tokenizer.tokenize(text)
    return len(tokens) / len(words)


## Part B — Self-attention with a padding mask
## Exercise 5 — Scaled dot-product attention with a mask


In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """Returns (output, weights). mask: bool, broadcastable to (B, H, T_q, T_k); True = may attend."""
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        scores = scores.masked_fill(mask == False, float('-inf'))
        
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, v)
    
    return output, weights


## Exercise 6 — The padding mask


In [ ]:
def make_padding_mask(input_ids, pad_id):
    """(B, T) token IDs -> (B, 1, 1, T) bool mask, True at non-padding positions."""
    return (input_ids != pad_id).unsqueeze(1).unsqueeze(2)


## Exercise 7 — Prove that padding cannot leak


In [ ]:
@torch.no_grad()
def padding_leakage(model, ids, junk_ids, pad_id, use_mask=True):
    """Max |output difference| at real positions when only padded positions change."""
    mask = make_padding_mask(ids, pad_id) if use_mask else None
    
    out_real = model(ids, mask)
    out_junk = model(junk_ids, mask)
    
    real_mask = (ids != pad_id).unsqueeze(-1)
    diff = torch.abs(out_real - out_junk)
    
    return diff[real_mask.expand_as(diff)].max().item()


## Part C — BERT: inputs, objective, and predictions
## Exercise 8 — BERT's three input tensors


In [ ]:
def describe_bert_input(enc, tokenizer):
    """Summarize one encoded sentence pair (batch size 1)."""
    ids, types, attn = enc["input_ids"][0], enc["token_type_ids"][0], enc["attention_mask"][0]
    ids_list = ids.tolist()
    
    return {
        "cls_position": ids_list.index(tokenizer.cls_token_id),
        "sep_positions": [i for i, val in enumerate(ids_list) if val == tokenizer.sep_token_id],
        "segment_b_tokens": int(((types == 1) & (attn == 1)).sum()),
        "pad_tokens": int((attn == 0).sum())
    }


## Exercise 9 — BERT's 80/10/10 masking rule


In [ ]:
def mask_tokens(input_ids, special_mask, mask_token_id, vocab_size, mlm_prob=0.15, generator=None):
    """BERT MLM masking. Returns (masked_ids, labels); labels are -100 where no prediction is required."""
    labels = input_ids.clone()
    masked_ids = input_ids.clone()
    
    prob_matrix = torch.full(labels.shape, mlm_prob)
    prob_matrix.masked_fill_(special_mask, 0.0)
    selected = torch.bernoulli(prob_matrix, generator=generator).bool()
    
    labels[~selected] = -100
    
    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8), generator=generator).bool() & selected
    masked_ids[indices_replaced] = mask_token_id
    
    indices_random = torch.bernoulli(torch.full(labels.shape, 0.5), generator=generator).bool() & selected & ~indices_replaced
    random_words = torch.randint(vocab_size, labels.shape, dtype=torch.long, generator=generator)
    masked_ids[indices_random] = random_words[indices_random]
    
    return masked_ids, labels


## Exercise 10 — BERT's top-k predictions


In [ ]:
@torch.no_grad()
def top_k_predictions(text, k=5):
    """[(token, probability), ...] for the single [MASK] in text, most probable first."""
    enc = bert_tok(text, return_tensors="pt").to(device)
    logits = bert_mlm(**enc).logits
    
    mask_idx = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0].item()
    probs = F.softmax(logits[0, mask_idx], dim=-1)
    
    topk_probs, topk_indices = torch.topk(probs, k)
    
    return [(bert_tok.convert_ids_to_tokens(idx.item()), prob.item()) for prob, idx in zip(topk_probs, topk_indices)]


## Exercise 11 — A careful bias probe


In [ ]:
@torch.no_grad()
def pronoun_probs(sentence):
    """(P('he'), P('she')) at the single [MASK] in sentence, from the full softmax over the vocabulary."""
    enc = bert_tok(sentence, return_tensors="pt").to(device)
    logits = bert_mlm(**enc).logits
    
    mask_idx = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero(as_tuple=True)[0].item()
    probs = F.softmax(logits[0, mask_idx], dim=-1)
    
    he_id = bert_tok.convert_tokens_to_ids("he")
    she_id = bert_tok.convert_tokens_to_ids("she")
    
    return (probs[he_id].item(), probs[she_id].item())


## Written Answers

**Question 1**
If `(("h", "u"), "hu")` is processed first, `hug` becomes `["hu", "g"]`. It never triggers the subsequent `(("u", "g"), "ug")` rule because the standalone `u` is no longer available. This demonstrates that subword splits are path-dependent; merges must be applied in the exact sequence they were learned to accurately reconstruct the frequency-based vocabulary.

**Question 2**
BPE allows for partial matches, meaning an unknown character only disrupts itself, leaving the rest of the string successfully tokenized. WordPiece's behavior acts as a strict guardrail, discarding the whole word to prevent garbage contexts from confusing the model. GPT-2's byte-level BPE doesn't need an `[UNK]` token because the base vocabulary contains all 256 possible bytes, ensuring any string can be fully represented.

**Question 3**
(a) As vocabulary size increases (e.g., 1000 to 16000), fertility decreases toward 1.0 because more complete words are stored as single tokens instead of being split into smaller fragments. 
(b) A 16,000-token vocabulary requires 12.2 million embedding parameters ($16,000 \times 768$), whereas a 1,000-token vocabulary only requires 768,000 parameters. 
(c) The trade-off is between sequence length (compute efficiency in attention) and embedding size (memory capacity). Measurements are taken on the validation set to ensure the tokenizer generalizes well to text it wasn't trained on.

**Question 4**
(a) `bert-base-uncased` is most efficient for English (lowest fertility), while `bert-base-multilingual-cased` is best for Hindi and Hinglish. `gpt2` is the least efficient for Hindi due to its byte-level fallback inflating the token count for Devanagari characters.
(b) The `bert-base-uncased` tokenizer completely strips Devanagari characters or maps them entirely to `[UNK]` tokens because it isn't trained for those scripts, effectively destroying the sentence content. 
(c) For users writing in Hindi, token-based APIs will be significantly more expensive and they will exhaust LLM context windows far faster than English writers.

**Question 5**
(a) IDs: `(2, 6)`. Embeddings: `(2, 6, 32)`. `q` split: `(2, 4, 6, 8)`. Attention weights: `(2, 4, 6, 6)`. Output: `(2, 6, 32)`.
(b) It is harmless because BERT discards predictions at padded positions during the loss calculation (by assigning them a `-100` label), meaning the gradients at these positions don't affect model weights.
(c) BERT is bidirectional and is designed to allow all context to be read simultaneously, so it has no causal mask. GPT-2 is an autoregressive decoder that predicts the next token in sequence, so it requires a causal mask to prevent current tokens from looking at future tokens. Both use padding masks to ignore batch alignment filler.

**Question 6**
The first `[SEP]` belongs to segment A (token type 0), ending the first sentence. The second `[SEP]` belongs to segment B (token type 1). Padding positions typically receive token type 0, which doesn't matter because the `attention_mask` completely blocks them from interacting with real tokens. In a sentence-pair classification task (like Natural Language Inference), the `[CLS]` token's final hidden state is pooled and passed to a linear layer to serve as the aggregate representation for the entire pair.

**Question 7**
(a) Replacing 100% of targets with `[MASK]` causes a train-test mismatch because the `[MASK]` token never appears in normal text during downstream fine-tuning. 
(b) Keeping 10% unchanged encourages the model to trust the provided token and build continuous context, while replacing 10% with random tokens forces the model to heavily rely on surrounding context to detect anomalies.
(c) It is harmless because the random selection successfully challenges the model contextually. If it randomly guesses the correct word, the model predicting the correct word is functioning exactly as intended.

**Question 8**
Accuracy ranks as: `unchanged` > `[MASK]` > `random`. `unchanged` accuracy is exceptionally high because the model simply learns to copy the input token embeddings forward. `[MASK]` is moderately difficult because the model must infer the missing item purely from context. `random` is the most difficult because the model must overcome explicitly misleading inputs. Reporting `unchanged` accuracy is flawed because it largely measures the model's ability to copy existing data rather than its contextual understanding.

**Question 9**
(a) The charts demonstrate that the model associates certain professions with specific genders (e.g., predicting "she" for nurses and "he" for engineers). However, the preference heavily depends on the prompt; different templates slightly shift the exact probabilities and rankings.
(b) 1. A single template structure fails to represent the wide syntactic diversity of real language. 2. Pre-training mask probabilities do not perfectly map to the behavior of a fine-tuned downstream classification system. 3. Softmax probabilities at a single masking position fail to capture complex, multi-token bias relationships or phrase structures.
(c) The experiment would be more reliable if evaluated on established downstream coreference resolution datasets (like WinoBias) or averaged across hundreds of diverse, syntactically varied templates.
